# 🎬 MovieMate — RAG-Based Conversational Movie Chatbot

> **Project:** Build a Retrieval-Augmented Generation (RAG) chatbot that recommends movies using semantic search + a large language model.

---

**RAG Pipeline**

```
User Query
    │
    ▼
Query Embedding  (sentence-transformers: all-MiniLM-L6-v2)
    │
    ▼
FAISS Similarity Search  (IndexFlatIP, cosine similarity)
    │
    ▼
Top-5 Movie Context Retrieved
    │
    ▼
Claude LLM Response  (claude-sonnet-4-6)
    │
    ▼
Conversational Response
```

---
**Table of Contents**
1. [Dataset Exploration](#1)
2. [Exploratory Data Analysis](#2)
3. [Data Preprocessing](#3)
4. [Embedding and Retrieval](#4)
5. [Conversational Movie Chatbot](#5)
6. [Interactive Interface](#6)
7. [Evaluation and Reflection](#7)

## Setup

In [ ]:
# !pip install pandas numpy matplotlib seaborn scikit-learn sentence-transformers faiss-cpu anthropic gradio==3.50.2

import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

from src.data_loader  import load_movies
from src.preprocessor import preprocess_movies, add_text_representations
from src.embedder     import MovieEmbedder
from src.retriever    import MovieRetriever

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

print('Setup complete')

---
<a id='1'></a>
## 1. Dataset Exploration

### 1.1 Description of Dataset Features

The MovieMate dataset is a curated collection of **218 highly-rated films** spanning 1941–2023, selected to represent diverse genres, eras, and international cinema.

| Column | Type | Description |
|--------|------|-------------|
| `title` | string | Movie title |
| `year` | int | Release year |
| `rating` | float | IMDb rating (1–10) |
| `votes` | int | Number of IMDb votes |
| `genre` | string | Pipe-separated genre tags |
| `director` | string | Director name(s) |
| `cast` | string | Comma-separated lead actors |
| `duration_mins` | int | Runtime in minutes |
| `plot_summary` | string | Short plot description |
| `language` | string | Primary language |

In [ ]:
df_raw = load_movies()
print(f'Rows: {len(df_raw):,}  |  Columns: {df_raw.shape[1]}')
df_raw.head()

In [ ]:
print('Column data types:')
print(df_raw.dtypes)
print('\nNull counts per column:')
print(df_raw.isnull().sum())

### 1.2 Summary Statistics

In [ ]:
df_raw[['year', 'rating', 'votes', 'duration_mins']].describe().round(2)

In [ ]:
all_genres = [
    g.strip()
    for genres in df_raw['genre'].dropna()
    for g in str(genres).split('|')
]
genre_counts = Counter(all_genres)
print(f'Unique genres: {len(genre_counts)}')
print('\nTop 10 genres:')
for genre, count in genre_counts.most_common(10):
    bar = '█' * count
    print(f'  {genre:<18} {count:>3}  {bar}')

In [ ]:
print('Language distribution (top 10):')
print(df_raw['language'].value_counts().head(10).to_string())
print('\nMost frequent directors in dataset:')
print(df_raw['director'].value_counts().head(10).to_string())

### 1.3 Observations from the Dataset

- **High average rating (~8.1/10):** The dataset is deliberately curated for quality — not a random IMDb sample. This bias is intentional: we want to recommend genuinely good films.
- **Drama dominates:** Drama is the most common tag, often combined with Action, Crime, or Sci-Fi as compound genres.
- **Long tail of directors:** Most directors appear once. A few prolific auteurs (Nolan, Kubrick, Kurosawa, Villeneuve) have multiple entries reflecting consistent quality.
- **International coverage:** ~15% non-English films (Japanese, French, Korean, Iranian, Italian, German, Spanish).
- **Year range 1941–2023:** Peaks in 1990s–2010s. Pre-1960 classics are fewer in count but consistently top-rated.
- **Minimal missing data:** A few films have missing `cast` or `duration_mins` — handled in preprocessing.

---
<a id='2'></a>
## 2. Exploratory Data Analysis

### 2.1 Visualizations of Ratings

In [ ]:
df_raw['decade'] = (df_raw['year'] // 10 * 10).astype(str) + 's'

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df_raw['rating'].plot.hist(bins=20, color='#4f46e5', alpha=0.85, edgecolor='white', ax=axes[0])
axes[0].axvline(df_raw['rating'].mean(), color='#ef4444', linestyle='--', linewidth=2,
                label=f"Mean = {df_raw['rating'].mean():.2f}")
axes[0].set_title('IMDb Rating Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

decade_order = sorted(df_raw['decade'].unique())
sns.boxplot(data=df_raw, x='decade', y='rating', order=decade_order, palette='viridis', ax=axes[1])
axes[1].set_title('Rating by Decade', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Decade')
axes[1].set_ylabel('Rating')
axes[1].tick_params(axis='x', rotation=45)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

### 2.2 Visualizations of Genres

In [ ]:
top_genres = pd.Series(all_genres).value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 5))
colors = sns.color_palette('muted', len(top_genres))
bars = ax.barh(top_genres.index[::-1], top_genres.values[::-1], color=colors[::-1], edgecolor='white')
for bar, val in zip(bars, top_genres.values[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10, color='#374151')
ax.set_title('Top 15 Genres (by number of movies tagged)', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Movies')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
genre_rating_rows = []
for _, row in df_raw.iterrows():
    for g in str(row['genre']).split('|'):
        genre_rating_rows.append({'genre': g.strip(), 'rating': row['rating']})

genre_df  = pd.DataFrame(genre_rating_rows)
genre_avg = (
    genre_df.groupby('genre')['rating']
    .agg(['mean', 'count'])
    .query('count >= 5')
    .sort_values('mean', ascending=False)
    .head(12)
)

fig, ax = plt.subplots(figsize=(12, 5))
palette = sns.color_palette('coolwarm_r', len(genre_avg))
ax.bar(genre_avg.index, genre_avg['mean'], color=palette, edgecolor='white', width=0.6)
ax.axhline(df_raw['rating'].mean(), color='#6b7280', linestyle='--', linewidth=1.5,
           label=f'Overall avg ({df_raw["rating"].mean():.2f})')
ax.set_ylim(7.5, 9.0)
ax.set_title('Average IMDb Rating by Genre  (genres with ≥5 movies)', fontsize=13, fontweight='bold')
ax.set_ylabel('Avg Rating')
ax.tick_params(axis='x', rotation=30)
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

### 2.3 Common Movie Trends

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

decade_counts = df_raw['decade'].value_counts().sort_index()
decade_counts.plot.bar(color='#6366f1', edgecolor='white', alpha=0.85, ax=axes[0])
axes[0].set_title('Movies per Decade', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Decade')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

dur_by_decade = df_raw.groupby('decade')['duration_mins'].mean().sort_index()
dur_by_decade.plot(marker='o', color='#f59e0b', linewidth=2.5, markersize=7, ax=axes[1])
axes[1].fill_between(range(len(dur_by_decade)), dur_by_decade.values, alpha=0.15, color='#f59e0b')
axes[1].set_xticklabels(dur_by_decade.index, rotation=45)
axes[1].set_title('Avg Runtime per Decade (minutes)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Duration (min)')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
scatter = ax.scatter(
    df_raw['votes'], df_raw['rating'],
    c=df_raw['year'], cmap='plasma',
    alpha=0.7, s=60, edgecolors='white', linewidths=0.4
)
plt.colorbar(scatter, ax=ax, label='Release Year')
ax.set_xscale('log')
ax.set_title('IMDb Rating vs. Vote Count  (colored by release year)', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Votes (log scale)')
ax.set_ylabel('Rating')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for _, row in df_raw.nlargest(5, 'rating').iterrows():
    ax.annotate(row['title'], (row['votes'], row['rating']),
                fontsize=8, xytext=(5, 3), textcoords='offset points', color='#1f2937')

plt.tight_layout()
plt.show()

---
<a id='3'></a>
## 3. Data Preprocessing

### 3.1 Cleaning and Preparing the Dataset

In [ ]:
df = preprocess_movies(df_raw)
print(f'After preprocessing: {len(df)} movies  (removed {len(df_raw) - len(df)} duplicates)')
df[['title', 'year', 'rating', 'genre', 'decade']].head(10)

### 3.2 Handling Missing Values

In [ ]:
before = df_raw.isnull().sum()
after  = df.isnull().sum()
comparison = pd.DataFrame({'Before': before, 'After': after})
comparison['Fixed'] = comparison['Before'] - comparison['After']
print('Missing values before and after preprocessing:')
print(comparison)

print('\nStrategies applied:')
strategies = [
    ('title / year duplicates', 'drop_duplicates(subset=[title, year])'),
    ('rating missing',          'fill with column median'),
    ('year missing',            'fill with 2000, cast to int'),
    ('duration_mins missing',   'fill with 0, cast to int'),
    ('votes missing',           'fill with 0, cast to int'),
    ('plot_summary missing',    'replace with "No summary available."'),
    ('genre parsing',           'split on | into genre_list column'),
    ('cast parsing',            'split on , keep top 3 actors'),
    ('decade column added',     '(year // 10 * 10) + "s"'),
]
for col, strategy in strategies:
    print(f'  {col:<28} → {strategy}')

### 3.3 Building Text Representations for Embedding

Each movie is serialised into a single rich text document combining all its metadata. This text is what gets embedded into a dense vector.

In [ ]:
df = add_text_representations(df)

example = df[df['title'] == 'Inception'].iloc[0]
print('Example text representation (what gets embedded):')
print('─' * 60)
print(example['text'])
print('─' * 60)
print(f'Length: {len(example["text"])} characters')

---
<a id='4'></a>
## 4. Embedding and Retrieval

### 4.1 Generating Vector Representations

We use **`sentence-transformers/all-MiniLM-L6-v2`** — a lightweight (~90MB) model that produces
high-quality 384-dimensional semantic embeddings. Vectors are L2-normalised so inner product = cosine similarity.

In [ ]:
embedder   = MovieEmbedder()
embeddings = embedder.fit_transform(df['text'].tolist())

print(f'Embedding matrix shape : {embeddings.shape}')
print(f'  {embeddings.shape[0]} movies  x  {embeddings.shape[1]} dimensions')
print(f'  dtype                : {embeddings.dtype}')
print(f'  L2-normalised        : {np.allclose(np.linalg.norm(embeddings, axis=1), 1.0)}')

In [ ]:
from sklearn.decomposition import PCA

df['primary_genre'] = df['genre'].apply(lambda g: str(g).split('|')[0].strip())
top_pg = [g for g, _ in Counter(df['primary_genre']).most_common(8)]

mask       = df['primary_genre'].isin(top_pg)
emb_sub    = embeddings[mask.values]
genres_sub = df.loc[mask, 'primary_genre'].values
titles_sub = df.loc[mask, 'title'].values

pca    = PCA(n_components=2, random_state=42)
emb_2d = pca.fit_transform(emb_sub)

palette = dict(zip(top_pg, sns.color_palette('tab10', len(top_pg))))

fig, ax = plt.subplots(figsize=(12, 7))
for g in top_pg:
    idx = genres_sub == g
    ax.scatter(emb_2d[idx, 0], emb_2d[idx, 1], label=g, alpha=0.75, s=55,
               color=palette[g], edgecolors='white', linewidths=0.4)

notable = ['Inception', 'The Dark Knight', 'The Godfather', 'Spirited Away',
           'The Shining', 'Interstellar', "Schindler's List", 'The Matrix']
for i, t in enumerate(titles_sub):
    if t in notable:
        ax.annotate(t, (emb_2d[i, 0], emb_2d[i, 1]), fontsize=7.5,
                    xytext=(5, 4), textcoords='offset points',
                    color='#1f2937', fontweight='semibold')

ax.set_title('Movie Embeddings — PCA 2D Projection  (coloured by primary genre)',
             fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(loc='lower right', fontsize=9, framealpha=0.85)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f'Total variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%')

### 4.2 Implementing Similarity Search

**FAISS** `IndexFlatIP` performs exact cosine-similarity nearest-neighbour search.
Because vectors are L2-normalised, dot product = cosine similarity — no additional normalisation needed at query time.

In [ ]:
retriever = MovieRetriever(df, embedder)
retriever.build_index(embeddings)

In [ ]:
query = 'mind-bending sci-fi movie with stunning visuals'
results = retriever.search(query, k=5)

print(f'Query: "{query}"\n')
print(f'{"#":<4} {"Title":<35} {"Year":<6} {"Rating":<8} {"Cosine Sim"}')
print('─' * 65)
for i, r in enumerate(results, 1):
    print(f'{i:<4} {r["title"]:<35} {int(r["year"]):<6} {r["rating"]:<8} {r["similarity_score"]:.4f}')

In [ ]:
# Hybrid: semantic + metadata filters
query2  = 'emotional family drama'
results2 = retriever.search_by_filter(
    query=query2, genre='Drama', min_rating=8.0, min_year=2000, k=5
)

print(f'Query: "{query2}"  |  genre=Drama  rating≥8.0  year≥2000\n')
print(f'{"#":<4} {"Title":<35} {"Year":<6} {"Rating":<8} {"Cosine Sim"}')
print('─' * 65)
for i, r in enumerate(results2, 1):
    print(f'{i:<4} {r["title"]:<35} {int(r["year"]):<6} {r["rating"]:<8} {r["similarity_score"]:.4f}')

In [ ]:
# Pairwise cosine similarity heatmap for a selection of films
sample_titles = [
    'Inception', 'Interstellar', 'The Matrix', 'Blade Runner',
    'The Godfather', 'Goodfellas', 'Pulp Fiction',
    'Spirited Away', 'Princess Mononoke', 'The Shining', 'Alien'
]

sample_df  = df[df['title'].isin(sample_titles)].copy()
sample_emb = embeddings[sample_df.index.tolist()]
sim_matrix = sample_emb.dot(sample_emb.T)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(sim_matrix,
            xticklabels=sample_df['title'].tolist(),
            yticklabels=sample_df['title'].tolist(),
            annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Cosine Similarity'})
ax.set_title('Pairwise Cosine Similarity — Sample Movies', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

---
<a id='5'></a>
## 5. Conversational Movie Chatbot

### 5.1 Integrating Retrieval with an LLM

The `MovieChatbot` class implements the full RAG loop:
1. Embed the user message with `sentence-transformers`
2. Retrieve top-5 movies from FAISS
3. Serialise movie metadata into a context block
4. Call the Claude API with system prompt + conversation history + context
5. Return the response and the retrieved movies

In [ ]:
from src.chatbot import MovieChatbot

# Show context injection pattern
sample_movies = retriever.search('action thriller', k=3)
context_block = '\n\n'.join([
    f"Movie {i+1}: {m['title']} ({int(m['year'])})\n"
    f"  Genre: {m['genre']} | Rating: {m['rating']}/10\n"
    f"  Director: {m['director']}\n"
    f"  Cast: {m['cast']}\n"
    f"  Plot: {m['plot_summary']}"
    for i, m in enumerate(sample_movies)
])
print('RAG context block injected into LLM prompt (example):')
print('─' * 60)
print(context_block[:900] + '...')

### 5.2 Generating Natural Language Responses

> **Note:** The cells below call the Claude API. Set `ANTHROPIC_API_KEY` to run them in live mode.

In [ ]:
HAS_KEY = bool(os.environ.get('ANTHROPIC_API_KEY'))

if HAS_KEY:
    chatbot = MovieChatbot(retriever, top_k=5)
    q1 = 'Recommend a mind-bending sci-fi film with great visuals'
    response1, retrieved1 = chatbot.chat(q1)
    print(f'User: {q1}\n')
    print(f'MovieMate:\n{response1}\n')
    print(f'[FAISS retrieved: {[r["title"] for r in retrieved1]}]')
else:
    print('ANTHROPIC_API_KEY not set — showing pre-written demo response\n')
    print('User: Recommend a mind-bending sci-fi film with great visuals\n')
    print('MovieMate:')
    print('Great choice of genre! Based on the database, I highly recommend **Inception** (2010)')
    print('by Christopher Nolan — a layered dream-heist thriller rated 8.8/10.')
    print('The rotating hallways and folding cities are visually unlike anything else.')
    print('You might also enjoy **Interstellar** (2014) for awe-inspiring space sequences,')
    print('or **Blade Runner 2049** (2017) for gorgeous neo-noir atmospheric visuals.')
    print('\n[FAISS retrieved: Inception, Interstellar, Blade Runner 2049, The Matrix, Arrival]')

In [ ]:
if HAS_KEY:
    # Multi-turn: follow-up question
    q2 = 'What about something from the 1980s?'
    response2, retrieved2 = chatbot.chat(q2)
    print(f'User: {q2}\n')
    print(f'MovieMate:\n{response2}\n')
    print(f'[FAISS retrieved: {[r["title"] for r in retrieved2]}]')
    chatbot.reset()
else:
    print('User: What about something from the 1980s?\n')
    print('MovieMate:')
    print('For 1980s sci-fi visuals, **Blade Runner** (1982) by Ridley Scott is essential.')
    print('The HR Giger-inspired neo-noir cityscape defined a generation.')
    print('**Alien** (1979) — just before the 80s — is another masterclass in atmosphere.')
    print('Both show that stunning visuals long predate CGI.')
    print('\n[FAISS retrieved: Blade Runner, Alien, The Thing, 2001: A Space Odyssey, Aliens]')

---
<a id='6'></a>
## 6. Interactive Interface

### 6.1 Simple Web Interface for User Interaction

The MovieMate web UI is built with **Gradio 3.50.2** and exposes four tabs:

| Tab | Description |
|-----|-------------|
| 💬 **Chat** | Multi-turn conversational recommendations via RAG + Claude |
| 🔍 **Semantic Search** | Natural language search with genre / rating / year filters |
| 📊 **Dataset Explorer** | Filterable, sortable table of all 218 movies |
| ℹ️ **About** | Architecture diagram, tech stack, setup instructions |

**Key design decisions:**
- Year mentions in the query text ("from 2021", "after 2019") are parsed and auto-converted to year filters
- Each genre has its own accent colour in movie cards
- FAISS relevance score rendered as a progress bar per card
- Demo mode when `ANTHROPIC_API_KEY` is not set; live Claude API responses when it is

In [ ]:
print('To launch the full MovieMate interface:')
print('  $ python demo_app.py')
print('  Open: http://localhost:7860')
print()
print('For live Claude API responses:')
print('  $ export ANTHROPIC_API_KEY="sk-ant-..."')
print('  $ python demo_app.py')

In [ ]:
# Minimal inline Gradio chatbot demo
import gradio as gr

_DEMO = [
    'Based on your query, here are top recommendations from the database:\n\n'
    '1. **Inception (2010)** ⭐ 8.8 — Mind-bending dream-heist by Nolan.\n'
    '2. **Interstellar (2014)** ⭐ 8.6 — Wormhole time-dilation epic.\n'
    '3. **The Matrix (1999)** ⭐ 8.7 — Simulated reality action thriller.\n\n'
    '*Want to narrow down by year, language, or runtime?*',
]
_n = {'i': 0}

def notebook_chat(user_msg, history):
    if HAS_KEY:
        bot = MovieChatbot(retriever, top_k=5)
        reply, _ = bot.chat(user_msg)
    else:
        reply = _DEMO[_n['i'] % len(_DEMO)]
        _n['i'] += 1
    history.append((user_msg, reply))
    return '', history

with gr.Blocks(title='MovieMate') as mini:
    gr.Markdown('### 🎬 MovieMate — Notebook Demo')
    cb = gr.Chatbot(height=340)
    tb = gr.Textbox(placeholder='Ask for a movie recommendation…', label='')
    tb.submit(notebook_chat, [tb, cb], [tb, cb])

mini.launch(inline=True, height=430, quiet=True)

---
<a id='7'></a>
## 7. Evaluation and Reflection

### 7.1 Retrieval Quality — Recall Evaluation

In [ ]:
eval_queries = [
    ('A thief who enters peoples dreams to steal secrets',              'Inception'),
    ('Rise of the Corleone mafia family in America',                   'The Godfather'),
    ('Astronauts travel through a wormhole to save humanity',          'Interstellar'),
    ('Man hunts wife killer using polaroid notes due to memory loss',  'Memento'),
    ('A girl works in a spirit bathhouse to free her parents',         'Spirited Away'),
    ('Two convicts bond over years in a brutal prison',                'The Shawshank Redemption'),
    ('Lawyer defends a black man wrongly accused in the South',        'To Kill a Mockingbird'),
    ('Humans enslaved in a simulated reality by machines',             'The Matrix'),
]

h1 = h3 = h5 = 0
print(f'{"Query":<55} {"Target":<32} {"@1":<4} {"@3":<4} {"Rank"}')
print('─' * 105)

for query, target in eval_queries:
    res    = retriever.search(query, k=5)
    titles = [r['title'] for r in res]
    rank   = next((i+1 for i, t in enumerate(titles) if target.lower() in t.lower()), None)
    
    e1 = '✅' if rank == 1       else '  '
    e3 = '✅' if rank and rank<=3 else '  '
    e5 = '✅' if rank and rank<=5 else '  '
    if rank == 1:       h1 += 1
    if rank and rank<=3: h3 += 1
    if rank and rank<=5: h5 += 1
    
    print(f'{query[:53]:<55} {target:<32} {e1:<4} {e3:<4} {rank or "—"}')

n = len(eval_queries)
print('─' * 105)
print(f'Recall@1: {h1}/{n} = {h1/n*100:.0f}%')
print(f'Recall@3: {h3}/{n} = {h3/n*100:.0f}%')
print(f'Recall@5: {h5}/{n} = {h5/n*100:.0f}%')

In [ ]:
# Similarity score distribution: signal-to-noise for different query types
test_queries = [
    'romantic comedy',
    'dark psychological thriller',
    'animated family adventure',
    'historical war epic',
]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes = axes.flatten()

for i, q in enumerate(test_queries):
    q_vec    = embedder.encode_query(q).astype('float32')
    all_sims = embeddings.dot(q_vec)
    top5     = sorted(all_sims, reverse=True)[:5]

    axes[i].hist(all_sims, bins=30, color='#6366f1', alpha=0.7, edgecolor='white')
    for s in top5:
        axes[i].axvline(s, color='#ef4444', linewidth=1.2, alpha=0.8)
    axes[i].set_title(f'"{q}"', fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Cosine Similarity')
    axes[i].set_ylabel('Count')
    axes[i].spines['top'].set_visible(False)
    axes[i].spines['right'].set_visible(False)

fig.suptitle('Similarity Score Distribution  (red lines = top-5 retrieved)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 7.2 Observations About Chatbot Performance

| Aspect | Observation |
|--------|-------------|
| **Semantic understanding** | Paraphrase robustness is strong — "corporate espionage in dreams" correctly retrieves *Inception* despite zero keyword overlap |
| **Genre boundary softness** | A Sci-Fi query may retrieve Thriller films with sci-fi elements — desirable since users often want cross-genre recommendations |
| **Multi-turn coherence** | With the Claude API active, follow-ups ("and a shorter one?") work because full history is sent on every turn |
| **Year filtering** | Hard year filters (sliders) are accurate; natural language years ("from 2021") are parsed via regex and auto-applied |
| **Demo mode** | Pre-written responses cover the 4 most common intents but are static — unusual queries get mismatched answers |

### 7.3 Limitations of the Current Approach

| Limitation | Detail | Possible Improvement |
|------------|--------|----------------------|
| **Small dataset** | 218 movies vs millions on IMDb | Expand with full IMDb or MovieLens dataset |
| **Static embeddings** | Rebuilt on every startup; no incremental updates | Incremental FAISS index additions |
| **No personalisation** | Same results for all users | Store per-session ratings / watch history |
| **Plot-only context** | No audience reception, box office, or critic signals | Add review embeddings or Metacritic scores |
| **English-centric model** | `all-MiniLM-L6-v2` is primarily English-trained | Use `paraphrase-multilingual-MiniLM-L12-v2` |
| **LLM hallucination risk** | Claude may confuse details not in the retrieved context | Stronger grounding instructions in the system prompt |
| **No result diversification** | FAISS returns cosine-ranked results without variety | Add MMR (Maximal Marginal Relevance) re-ranking |

### 7.4 Summary

MovieMate demonstrates that a RAG architecture is well-suited for knowledge-grounded conversational AI.
The combination of:
- A **small, high-quality curated dataset**
- **Dense retrieval** via sentence-transformers + FAISS
- **Grounded generation** via Claude with injected context

...produces specific, accurate, and engaging movie recommendations — a significant improvement over a bare LLM prompt, which can hallucinate titles or give generic suggestions not grounded in any real database.

In [ ]:
print('=' * 60)
print('  MovieMate — Final Project Summary')
print('=' * 60)
print(f'  Dataset       : {len(df)} movies  ({df["year"].min()}–{df["year"].max()})')
print(f'  Avg Rating    : {df["rating"].mean():.2f} / 10')
genres_all = set(g.strip() for gs in df['genre'].dropna() for g in str(gs).split('|'))
print(f'  Genres        : {len(genres_all)} unique')
print(f'  Embedding dim : {embeddings.shape[1]}')
print(f'  Model         : sentence-transformers/all-MiniLM-L6-v2')
print(f'  Vector store  : FAISS IndexFlatIP (cosine similarity)')
print(f'  LLM           : claude-sonnet-4-6 (Anthropic)')
print(f'  Interface     : Gradio 3.50.2')
print(f'  API mode      : {"Live" if HAS_KEY else "Demo (set ANTHROPIC_API_KEY to enable)"}')
print('=' * 60)